
# NeoOLAF native DocRED layer ablation — Skai TV v5

This notebook runs **one complete native NeoOLAF Layer 0–12 pipeline** on one
DocRED document. It changes no file under `src/neoolaf`.

v5 focuses on the four remaining issues from v4:

1. the single whole-document Layer 1 call performs an explicit country-coverage
   check and emits separate ORG/LOC → country relation instances when supported;
2. Layer 2 receives only a compact top-five ontology shortlist, relevant rules,
   and at most two relevant synthetic examples;
3. transparent profile guardrails distinguish P127/P749/P361,
   P159/P276/P131, P571/P577, and P17/P27 without creating new relations;
4. after execution only, audited mention projection maps benchmark variants such
   as `Athens metropolitan area` to the DocRED `Athens` cluster.

The country relations must originate in the native Layer 1 output. The evaluator
never creates relations. No direct DocRED extractor, source-entity anchoring,
gold relation hints, or post-run closure is used.


In [ ]:

from __future__ import annotations

import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_ablation_v5 import (
    analyze_run,
    load_layer_states,
    project_values_to_gold,
    read_json,
    read_jsonl,
    run_native_pipeline,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


## 1. Configuration

In [ ]:

INPUT_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_input_v5.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_gold.jsonl"
ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/docred_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "skai_tv_country_compact_v5"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# All Layer 2 relation decisions can run in one concurrent wave for this document.
WORKERS = 16
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


## 2. Preflight: anti-leakage, ontology and compact prompts

In [ ]:

required = [
    INPUT_JSONL, GOLD_JSONL, ONTOLOGY_PATH, ONTOLOGY_ORIGINAL,
    RELATION_CATALOG, RELATION_ALIASES, PROFILE_PATH, GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]

profile = read_json(PROFILE_PATH)
task = input_rows[0]["task_guidance"]
catalog = read_json(RELATION_CATALOG)

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Ontology properties:", catalog["property_count"])
print("Allowed ontology IDs:", len(task["allowed_relation_ids"]))
print("Country propagation examples:", len(task["country_propagation_examples"]))
print("Predicate-specific candidate sets:", len(task["predicate_candidate_sets"]))
print("Layer 2 workers:", profile["layers"]["layer02_candidate_enrichment"]["max_concurrency"])
print("Layer 2 property cap:", profile["layer02_compact_candidate_cap"])
print("Mention-free relation schemas injected:", len(profile["relations"]["allowed"]))

assert len(task["allowed_relation_ids"]) == 96
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False


## 3. Run the full native pipeline

In [ ]:

if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)



### Runtime evidence saved

- `run_logs/layer01_relation_instances.json`
- `run_logs/layer01_country_coverage_audit.json`
- `run_logs/layer02_contrastive_decisions.json`
- `run_logs/layer02_compact_prompt_audit.json`
- `run_logs/layer04_endpoint_assignment.json`
- `run_logs/layer04_constraint_rejections.json`
- `run_logs/llm_calls.jsonl`
- `run_logs/llm_errors.jsonl`
- `run_logs/llm_parse_errors.jsonl`
- `run_logs/ontology_retrieval.jsonl`
- raw responses under `run_logs/responses/`
- `analysis/gold_relation_trace_v5.csv`
- `analysis/entity_projection_audit_v5.csv`


## 4. Strict evaluation and cumulative ablation

In [ ]:

summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

display(pd.DataFrame(summary["layer_summary"]))

print("Strict final native evaluation with audited v5 entity projection")
display(pd.DataFrame([{
    key: summary["strict_evaluation_v5"][key]
    for key in [
        "predicted", "gold", "true_positive", "false_positive",
        "false_negative", "precision", "recall", "f1",
    ]
}]))

print("Earlier projection result retained for comparison")
display(pd.DataFrame([{
    key: summary["strict_evaluation_before_v5_projection"][key]
    for key in ["predicted", "gold", "true_positive", "precision", "recall", "f1"]
}]))

print("Cumulative strict evaluation v5")
display(pd.DataFrame(summary["cumulative_strict_evaluation_v5"]))

print("v5 first-failure counts")
print(summary["failure_counts_v5"])


## 5. Country coverage and relation-instance extraction

In [ ]:

layer1_decisions = read_json(RUN_DIR / "run_logs/layer01_relation_instances.json")
country_audit = read_json(RUN_DIR / "run_logs/layer01_country_coverage_audit.json")

display(pd.DataFrame(layer1_decisions))
print("Country relation instances emitted by the single Layer 1 call:", country_audit["country_relation_instance_count"])
display(pd.DataFrame(country_audit["country_relation_instances"]))


## 6. Compact Layer 2 decisions and profile guardrails

In [ ]:

layer2_decisions = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_contrastive_decisions.json"))
compact_audit = pd.DataFrame(read_json(RUN_DIR / "run_logs/layer02_compact_prompt_audit.json"))

display(layer2_decisions)
if not layer2_decisions.empty:
    print("Guardrail overrides:", int(layer2_decisions["guardrail_override"].fillna(False).sum()))
    display(layer2_decisions[layer2_decisions["guardrail_override"].fillna(False)])

print("Compact prompt audit")
display(compact_audit)
if not compact_audit.empty:
    print("Mean Layer 2 user characters:", round(compact_audit["user_chars"].mean(), 1))
    print("Maximum Layer 2 user characters:", int(compact_audit["user_chars"].max()))
    print("Maximum candidate properties:", max(len(x) for x in compact_audit["candidate_relation_ids"]))


## 7. Mention-aware benchmark projection

In [ ]:

projection = pd.read_csv(RUN_DIR / "analysis/entity_projection_audit_v5.csv")
display(projection)

# Demonstrate the intended deterministic evaluation-only mapping.
gold = read_jsonl(GOLD_JSONL)[0]
print(project_values_to_gold(["Athens metropolitan area"], gold))


## 8. Correct gold-relation trace

In [ ]:

trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace_v5.csv")
display(trace_df)
display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)


## 9. Native candidates, assertions and triples

In [ ]:

states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer11 = states.get(11)

if layer3:
    display(pd.DataFrame([{
        "candidate_id": c.candidate_id,
        "canonical_label": c.canonical_label,
        "mentions": [m.text for m in c.mentions],
        "controlled_hints": [h for h in c.ontology_hints if str(h).lower().startswith("controlled_relation:")],
    } for c in layer3.relation_candidates or []]))
    assert all(c.mentions for c in layer3.relation_candidates or []), "Mention-free relation candidate detected."

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 triples")
    display(pd.DataFrame([{
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Layer 6 ontology relation candidates:", len(layer6.ontology_relation_candidates or []) if layer6 else None)
print("Layer 11 completion candidates:", len(layer11.completion_candidates or []) if layer11 else None)


## 10. Speed, concurrency and errors

In [ ]:

def optional_jsonl(path: Path) -> pd.DataFrame:
    return pd.DataFrame(read_jsonl(path)) if path.is_file() else pd.DataFrame()

calls = optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
retrieval = optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_system_chars=("system_chars", "mean"),
        mean_user_chars=("user_chars", "mean"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("Backend/API errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
if not retrieval.empty:
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))

manifest = read_json(RUN_DIR / "run_manifest.json")
print("Wall-clock pipeline seconds:", manifest.get("elapsed_seconds"))



## Success checklist

Before moving to the other four documents, verify that:

1. Layer 1 emits separate country relations for supported organizations and locations.
2. `Skai TV -> Skai Group` is canonicalized to P127 unless explicit branch/subsidiary wording exists.
3. the organization relaunch location is evaluated as P159, while the date remains P571.
4. `Athens metropolitan area` maps to the Athens gold cluster only in the evaluation projection.
5. each Layer 2 prompt contains at most five properties and is far smaller than v4.
6. Layer 4 exact endpoint resolution still requires no extra LLM call when endpoints resolve.
7. no country relation is created by post-run closure.
